# Dataset Validation (Shards) – Short Documentation

## 1. Purpose
Quickly confirm that the freshly generated position dataset (CSV shards) is qualitatively stable before: feature engineering, baseline training, building the “gold subset”.

---

## 2. Input
- Folder with files: `positions_shard_*.csv`
- Key columns: `fen, eval_cp, phase_bucket, time_ms, depth, material_white, material_black`
- Static parameters in code: paths, sample limits, `CLAMP_CP`.

---

## 3. Main Script Tasks
1. Stream shards in chunks (low RAM footprint).
2. Aggregate phase counts: OPEN / MID / END.
3. Count clamped evaluations (|eval_cp| == CLAMP_CP).
4. Collect limited samples for histograms: eval_cp, time_ms, depth, material difference.
5. Exclude depth == -1 from plots (still report share).
6. Generate PNG plots + text `summary.txt` with statistics.

---

## 4. Why These Metrics
- Phase distribution – detects sampling bias (overweighting one phase harms generalization).
- clamp_ratio – shows if clamping truncates evaluation tails (information loss on large advantages).
- Eval percentiles (p50/p90/p99) – shape, spread, and outliers of the target distribution.
- Time_ms – engine stability (especially p90/p99).
- Depth – actual search depth of “fast eval”; too shallow increases noise.
- Material diff – checks diversity (not only endgames or only openings).

---

## 5. Interpretation of Sample Results
- Phases: 30% / 30% / 40% → END overweight; acceptable if intentional (endgame focus). Otherwise, reduce endgame sampling.
- clamp_ratio ≈ 0.15% → very low, clamping not a constraint.
- Eval p99 = 552 cp << 2000 → large headroom; current clamp is fine.
- Time: mean ~15 ms, p99 ~48 ms → stable, no heavy tail.
- Depth: mean ≈10, p99 = 14 → expected for quick search; deeper “gold subset” needed later.
- Overall: dataset ready for baseline training.

---

## 6. Heuristic Thresholds (Guidelines)
- clamp_ratio: OK <3% | warn 3–6% | alert >6%.
- END share (if aiming ~33% balance): OK ≤38–40% | warn 40–45% | alert >45%.
- depth -1: OK <0.5% | warn 0.5–1% | alert >1%.
- Eval p99 near CLAMP_CP + rising clamp_ratio → consider higher clamp or “saturation” flag.

---

## 7. Deliberately Omitted
- FEN legality validation.
- Duplicate detection (no FEN set in this version).
- `eval_norm` consistency check.
- Correlation analysis (depth ↔ time ↔ eval).
- Side-to-move segmentation.

---

## 8. Potential Risks and Effects
1. Phase imbalance → model biased toward dominant phase.
2. High clamp_ratio → flattened targets, poorer learning on large swings.
3. Low depth → noisier labels, lower performance ceiling.
4. Unstable times (heavy tail) → inconsistent label quality.
5. Skewed material diff → model overfits narrow position types.

---

## 9. Minimal Next Steps
1. Add `eval_norm` validation (|eval_cp / CLAMP_CP − eval_norm| < 1e-3).
2. Count unique reduced FEN; report duplicates.
3. Generate JSON manifest (per shard: count, min/max eval, phase counts).
4. Train baseline MLP and log MAE_cp vs eval distribution percentiles.
5. Build “gold subset” (deeper search / MultiPV) to assess systematic bias.

---

## 10. Short Conclusion
Snapshot looks healthy (low saturation, stable times, moderate depth, controlled phase skew). Proceed to modeling while adding duplicate + eval_norm checks.

---

## 11. Plots

![Phase Distribution](../../plots/check_data_imbalance/phase_distribution.png)
![Eval Histogram](../../plots/check_data_imbalance/eval_hist.png)
![Eval CDF](../../plots/check_data_imbalance/eval_cdf.png)
![Time Histogram](../../plots/check_data_imbalance/time_hist.png)
![Time Log Histogram](../../plots/check_data_imbalance/time_hist_log.png)
![Depth Histogram](../../plots/check_data_imbalance/depth_hist.png)
![Material Difference Histogram](../../plots/check_data_imbalance/material_diff_hist.png)

---